# מטלת סיום – חלק 2 | Movie Rating Prediction

**שמות חברי הצוות + ת.ז.:** <FILL IN>

**קישור ל-GitHub:** <FILL IN>

In [ ]:
# ============================================================
#  מטלת סיום – חלק 2 | Movie Rating Prediction
#  Chen Hajaj · Ariel University · Machine Learning
# ============================================================
# שמות חברי הצוות + ת.ז.:  <FILL IN>
# קישור ל-GitHub:            <FILL IN>
# ============================================================

## 1. ייבוא ספריות

In [1]:
import re
import ast
import bisect
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.linear_model    import ElasticNet
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.compose         import ColumnTransformer
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base            import BaseEstimator, TransformerMixin

## 2. טעינת הדאטה

In [2]:
dataset = pd.read_csv("dataset.csv", na_values=['\\N', 'N/A', 'null'], low_memory=False)
# טעינת קובץ של הבמאים 
df_crew = pd.read_csv("title.crew.tsv.gz", sep='\t', usecols=['tconst', 'directors'], low_memory=False)
# מיזוג של הקבצים לדאטה שכוללת את הבמאים 
dataset = pd.merge(dataset, df_crew, on='tconst', how='left')

## 3. בניית פונקציות עזר לניקוי הנתונים 


In [3]:
# ── פונקציות ניקוי ───────────────────────────────────────────
 
def _clean_year(val):
    """ערך תקין: רצף של 4 ספרות ."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.fullmatch(r'\d{4}', s)
    if m:
        year = int(s)
        return year if 1900 <= year <= 2024 else np.nan
    return np.nan
 
 
def _clean_runtime(val):
    """ערך תקין: מספר בלבד (לא כחלק מטקסט)."""
    if pd.isna(val):
        return np.nan
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan
 
 
def _clean_rating(val):
    """ערך תקין: מספר בין 1.0 ל-10.0."""
    if pd.isna(val):
        return np.nan
    try:
        r = float(val)
        return r if 1.0 <= r <= 10.0 else np.nan
    except (ValueError, TypeError):
        return np.nan
 
 
def _clean_genres(val):
    """מחזיר רשימת ז'אנרים נקייה."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    if re.match(r'^(\\N|N/A|NA|\[\])$', s, re.IGNORECASE):
        return []
    try:
        parsed = ast.literal_eval(s)
        items  = [str(g).strip() for g in parsed] \
                 if isinstance(parsed, list) else [str(parsed).strip()]
    except (ValueError, SyntaxError):
        items = re.split(r'[,|;]', s)
    cleaned = []
    for item in items:
        g = item.strip().title()
        if g and re.match(r'^[A-Za-z\- ]{2,30}$', g):
            if not re.match(r'^(N/A|\\N|Nan|None|Na)$', g, re.IGNORECASE):
                cleaned.append(g)
    return cleaned
 
 
def _clean_actors(val):
    """מחלץ מזהי nm תקינים בלבד."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    if re.match(r'^(\\N|\[\]|N/A)$', s, re.IGNORECASE):
        return []
    return re.findall(r'nm\d{7,8}', s)
 
 
def _clean_director(val):
    """מסיר sentinel בלבד, מחזיר כמו שהוא."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if re.match(r'^(\\N|N/A|NA|nan|none)$', s, re.IGNORECASE):
        return np.nan
    return s

## 4. Feature Engineering


In [42]:
##פונקציית עזר כללית לטיפול ברשימות
def _safe_list(val):
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [x.strip() for x in str(val).split(',') if x.strip()]

##הוספת הפיצ'רים הפשוטים - שלא מבוססים על נתוני עבר
def add_static_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['num_genres']  = df['genres'].apply(lambda g: len(_safe_list(g)))
    df['is_too_short'] = (df['runtimeMinutes'] < 75).astype(float)
    df['is_too_long']  = (df['runtimeMinutes'] > 150).astype(float)
    return df

## חישוב היחס בין אורך הסרט לחציון הסרטים עפי גאנר - רק סרטים של שנה קטנה מהשנה הנוכחית  
class RelativeRuntimeTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        df['_first_genre'] = df['genres'].apply(
            lambda g: _safe_list(g)[0] if _safe_list(g) else 'Unknown'
        )
        self.train_df_ = df[['_first_genre', 'startYear', 'runtimeMinutes']].copy()
        return self

    def transform(self, X):
        X = X.copy()
        X['_first_genre'] = X['genres'].apply(
            lambda g: _safe_list(g)[0] if _safe_list(g) else 'Unknown'
        )
        def calc(row):
            past = self.train_df_[
                (self.train_df_['_first_genre'] == row['_first_genre']) &
                (self.train_df_['startYear']    <  row['startYear'])
            ]
            median = past['runtimeMinutes'].median()
            if pd.isna(median) or median == 0:
                return np.nan
            return row['runtimeMinutes'] / median

        X['relative_runtime_by_genre'] = X.apply(calc, axis=1)
        return X.drop(columns=['_first_genre'])

## חישוב דירוג ממוצע עפי במאי - רק סרטים של שנה קטנה מהשנה הנוכחית        
class DirectorAvgTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X[['directors', 'startYear']].copy()
        df['averageRating'] = y
        self.train_df_ = (
            df.dropna(subset=['directors', 'startYear', 'averageRating'])
            .copy()
        )
        return self

    def transform(self, X):
        X = X.copy()
        def calc(row):
            past = self.train_df_[
                (self.train_df_['directors'] == row['directors']) &
                (self.train_df_['startYear'] <  row['startYear'])
            ]
            return past['averageRating'].mean()
        X['director_past_avg'] = X.apply(calc, axis=1)
        return X

##חישוב כמה פעמים שחקני הסרט שיחקו בעבר 
class ActorExpTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        actor_years = {}
        for _, row in X[['lead_actors_ids', 'startYear']].dropna().iterrows():
            for actor in _safe_list(row['lead_actors_ids']):
                actor_years.setdefault(actor, []).append(row['startYear'])
        self.actor_years_ = {
            a: sorted(years) for a, years in actor_years.items()
        }
        return self

    def transform(self, X):
        X = X.copy()
        def calc(row):
            actors = _safe_list(row['lead_actors_ids'])
            if not actors:
                return 0.0
            year = row['startYear']
            exp  = [bisect.bisect_left(self.actor_years_.get(a, []), year)
                    for a in actors]
            return float(np.sum(exp))
        X['avg_actors_past_experience'] = X.apply(calc, axis=1)
        return X

## 4. prepare_data

In [46]:
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    #מימוש פונקציות העזר של הניקוי
    df['startYear']       = df['startYear'].apply(_clean_year)
    df['runtimeMinutes']  = df['runtimeMinutes'].apply(_clean_runtime)
    df['averageRating']   = df['averageRating'].apply(_clean_rating)
    df['genres']          = df['genres'].apply(_clean_genres)
    df['lead_actors_ids'] = df['lead_actors_ids'].apply(_clean_actors)
    df['directors']       = df['directors'].apply(_clean_director)

    #הוספת הפיצ'רים הפשטוים - אלה שלא מבוססים על נתוני העבר
    df = add_static_features(df)

    #הסרת עמודות לא רלוונטיות
    COLS_TO_DROP = [
    'numVotes', 'BoxOffice', 'plot',
    'tconst', 'primaryTitle', 'Language', 'Country', 'budget',
    ]
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

    return df

## 5. הכנת דאטת האימון



In [47]:
dataset_clean = prepare_data(dataset.copy())
dataset_clean = dataset_clean.dropna(subset=['averageRating']).reset_index(drop=True)

y = dataset_clean['averageRating'].values
X = dataset_clean.drop(columns=['averageRating'])

## 6. הגדרת Pipeline ומודלים

In [21]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, FEATURE_COLS),
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# מודל 1 – Elastic Net (חובה לפי המטלה)
elastic_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        ElasticNet(max_iter=10_000, random_state=42)),
])

# מודל 2 – Random Forest
# נימוק: מודל לא-לינארי, עמיד לפני outliers,
# לא מניח שום התפלגות על הפיצ'רים
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

## 7. לולאת CV ידנית

### למה לולאה ידנית?

בכל fold צריך לחשב את הפיצ'רים ההיסטוריים מחדש –
רק מנתוני ה-train של אותו fold.
sklearn לא מאפשר זאת בתוך `cross_val_predict` רגיל.

### מניעת Leakage בלולאה

```
fold i:
  df_tr = שורות האימון בלבד
  df_te = שורות הבדיקה בלבד

  _build_director_past_avg(df_tr, df_te)
  ← מחפש היסטוריה רק ב-df_tr
  ← df_te לא משפיע על החישוב
```

In [8]:
def run_cv(model_pipeline, dataset_clean, y_full, kf, model_name):
    """
    מריצה 10-Fold CV ידנית.
    בכל fold:
      1. מחשב פיצ'רים היסטוריים מ-train בלבד
      2. מחיל על test
      3. מאמן ומחזה
    """
    oof_preds    = np.zeros(len(y_full))
    fold_metrics = []

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(dataset_clean)):
        df_tr = dataset_clean.iloc[train_idx].copy()
        df_te = dataset_clean.iloc[test_idx].copy()

        # ── פיצ'רים היסטוריים – train הוא המקור ────────────────
        df_tr['director_past_avg']          = _build_director_past_avg(df_tr, df_tr)
        df_tr['avg_actors_past_experience'] = _build_actor_past_experience(df_tr, df_tr)

        df_te['director_past_avg']          = _build_director_past_avg(df_tr, df_te)
        df_te['avg_actors_past_experience'] = _build_actor_past_experience(df_tr, df_te)

        # ── X, y ────────────────────────────────────────────────
        X_train = df_tr[FEATURE_COLS].copy()
        X_test  = df_te[FEATURE_COLS].copy()
        y_train = y_full[train_idx]
        y_test  = y_full[test_idx]

        # ── אימון וחיזוי ────────────────────────────────────────
        model_pipeline.fit(X_train, y_train)
        preds              = model_pipeline.predict(X_test)
        oof_preds[test_idx] = preds

        # ── מדדים ───────────────────────────────────────────────
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae  = mean_absolute_error(y_test, preds)
        r2   = r2_score(y_test, preds)
        fold_metrics.append({'fold': fold_num+1,
                              'rmse': rmse, 'mae': mae, 'r2': r2})

    metrics_df = pd.DataFrame(fold_metrics)
    print(f"\n── {model_name} – 10-Fold CV ──────────────────────")
    print(f"  RMSE: {metrics_df['rmse'].mean():.4f} ± {metrics_df['rmse'].std():.4f}")
    print(f"  MAE:  {metrics_df['mae'].mean():.4f}  ± {metrics_df['mae'].std():.4f}")
    print(f"  R²:   {metrics_df['r2'].mean():.4f}  ± {metrics_df['r2'].std():.4f}")
    return oof_preds, metrics_df

## 8. הרצת CV

In [9]:
oof_en, metrics_en = run_cv(elastic_pipe, dataset_clean, y_full, kf, 'Elastic Net')
oof_rf, metrics_rf = run_cv(rf_pipe,      dataset_clean, y_full, kf, 'Random Forest')

summary = pd.DataFrame([
    {'Model': 'Elastic Net',
     'RMSE': metrics_en['rmse'].mean(), 'RMSE_std': metrics_en['rmse'].std(),
     'MAE':  metrics_en['mae'].mean(),  'MAE_std':  metrics_en['mae'].std(),
     'R²':   metrics_en['r2'].mean(),   'R²_std':   metrics_en['r2'].std()},
    {'Model': 'Random Forest',
     'RMSE': metrics_rf['rmse'].mean(), 'RMSE_std': metrics_rf['rmse'].std(),
     'MAE':  metrics_rf['mae'].mean(),  'MAE_std':  metrics_rf['mae'].std(),
     'R²':   metrics_rf['r2'].mean(),   'R²_std':   metrics_rf['r2'].std()},
])
print("\n── השוואת מודלים ──────────────────────────────")
print(summary.to_string(index=False))

NameError: name 'elastic_pipe' is not defined

## 9. ניתוח חשיבות פיצ'רים

In [10]:
# אימון על כל הדאטה לצורך feature importance
elastic_pipe.fit(X_full, y_full)
rf_pipe.fit(X_full,      y_full)

# Elastic Net – לפי מקדמים אבסולוטיים
en_coefs = elastic_pipe.named_steps['model'].coef_
en_imp   = pd.DataFrame({
    'Feature':     FEATURE_COLS,
    'Coefficient': np.abs(en_coefs),
    'Direction':   ['+' if c > 0 else '-' for c in en_coefs],
}).sort_values('Coefficient', ascending=False)

# Random Forest – לפי impurity
rf_imp = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Importance': rf_pipe.named_steps['model'].feature_importances_,
}).sort_values('Importance', ascending=False)

print("── Elastic Net ─────────────────────────────────")
print(en_imp.to_string(index=False))
print("\n── Random Forest ───────────────────────────────")
print(rf_imp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, imp_df, col, title in zip(
        axes,
        [en_imp, rf_imp],
        ['Coefficient', 'Importance'],
        ['Elastic Net – |Coefficient|', 'Random Forest – Feature Importance']):
    ax.barh(imp_df['Feature'][::-1], imp_df[col][::-1], color='steelblue')
    ax.set_title(title)
    ax.set_xlabel(col)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

NameError: name 'elastic_pipe' is not defined

## 10. ניתוח שגיאות (Error Analysis)

In [11]:
best_rmse_en = metrics_en['rmse'].mean()
best_rmse_rf = metrics_rf['rmse'].mean()
best_oof     = oof_rf if best_rmse_rf < best_rmse_en else oof_en
best_name    = 'Random Forest' if best_rmse_rf < best_rmse_en else 'Elastic Net'
print(f"המודל הטוב יותר: {best_name}")

errors   = y_full - best_oof
error_df = dataset_clean[['averageRating']].copy()
error_df['predicted'] = best_oof
error_df['error']     = errors
error_df['abs_error'] = np.abs(errors)

if 'primaryTitle' in dataset_clean.columns:
    error_df['title'] = dataset_clean['primaryTitle'].values

print("\n── Top 10 Overpredictions ──────────────────────")
print(error_df.nsmallest(10, 'error').to_string())
print("\n── Top 10 Underpredictions ─────────────────────")
print(error_df.nlargest(10, 'error').to_string())

outliers_en = set(np.where(np.abs(y_full - oof_en) > 1.5)[0])
outliers_rf = set(np.where(np.abs(y_full - oof_rf) > 1.5)[0])
print(f"\noutliers Elastic Net:   {len(outliers_en)}")
print(f"outliers Random Forest: {len(outliers_rf)}")
print(f"חפיפה:                  {len(outliers_en & outliers_rf)}")

plt.figure(figsize=(6, 6))
plt.scatter(y_full, best_oof, alpha=0.3, color='steelblue')
plt.plot([1, 10], [1, 10], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title(f'Actual vs. Predicted – {best_name} (OOF CV)')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()

NameError: name 'metrics_en' is not defined

## 11. ניתוח הוגנות (Fairness Analysis)

In [12]:
def fairness_by_slice(y_true, y_pred, slice_series, min_size=30):
    df_f = pd.DataFrame({'y': y_true, 'pred': y_pred,
                         'slice': slice_series.values})
    rows = []
    for val, grp in df_f.groupby('slice'):
        if len(grp) < min_size:
            continue
        rmse = np.sqrt(mean_squared_error(grp['y'], grp['pred']))
        mae  = mean_absolute_error(grp['y'], grp['pred'])
        rows.append({'Slice': val, 'N': len(grp), 'RMSE': rmse, 'MAE': mae})
    return pd.DataFrame(rows).sort_values('RMSE', ascending=False)


decade_series = (dataset_clean['startYear'] // 10 * 10).astype('Int64')
fair_decade   = fairness_by_slice(y_full, best_oof, decade_series)
print("── Fairness: לפי עשור ──────────────────────────")
print(fair_decade.to_string(index=False))

if not fair_decade.empty:
    plt.figure(figsize=(9, 4))
    plt.bar(fair_decade['Slice'].astype(str), fair_decade['RMSE'], color='coral')
    plt.title('Fairness – RMSE by Decade')
    plt.ylabel('RMSE')
    plt.xlabel('Decade')
    plt.tight_layout()
    plt.savefig('fairness_decade.png', dpi=150)
    plt.show()

NameError: name 'dataset_clean' is not defined

## 12. שמירת מודל סופי + prepare_data

### prepare_data

מקבלת רק את דאטת התחרות.
ניגשת ל-`df_train_global` שנשמר בזיכרון ה-notebook –
כך הפיצ'רים ההיסטוריים מחושבים על בסיס דאטת האימון.

In [ ]:

# ── אימון סופי ─────────────────────────────────────────────────
final_model = rf_pipe if metrics_rf['rmse'].mean() < metrics_en['rmse'].mean() \
              else elastic_pipe
final_model.fit(X_full, y_full)
joblib.dump(final_model, 'model.pkl')
print(f"מודל נשמר: model.pkl ({best_name})")

# ── שימוש בתחרות ───────────────────────────────────────────────
# df_test = pd.read_csv('test.csv')
# X_test  = prepare_data(df_test)
# y_pred  = final_model.predict(X_test)

print("\n✅ סיום הרצת הקוד")